# NB02: generate OCDD v1

Selects baselines from the NB01 candidates, then writes for each one the processing-matched control (L0p), levels L1 to L5 (two variants each) and, for a quarter of them, single-flaw sweeps. Every clip passes the signal gates and a re-alignment drift gate (median ≤ 30 ms). Resumes where it stopped.

Runtime: T4 GPU (used for re-alignment).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/oratio")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/Jonathan-Jerrick/oratio.git", str(REPO)],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{REPO}/packages/speechgrade",
        "-e",
        f"{REPO}/dataset_factory[colab]",
    ],
    capture_output=True,
    text=True,
)
print(install.stdout[-2000:], install.stderr[-4000:])
install.check_returncode()
for src in (REPO / "packages/speechgrade/src", REPO / "dataset_factory/src"):
    if str(src) not in sys.path:
        sys.path.insert(0, str(src))
print(
    subprocess.run(
        ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True
    ).stdout
)

In [ ]:
from oratio_data import ctc_gpu
from oratio_data.stages import nb02

ROOT = Path("/content/drive/MyDrive/Oratio")
LIMIT = 16  # baselines; None for all
AUDIT_EVERY = 1 if ctc_gpu.has_gpu() else 4

In [ ]:
counts = nb02.generate(ROOT, ctc_gpu.best_runner(), limit=LIMIT, audit_every=AUDIT_EVERY)
print(counts)

In [ ]:
import pandas as pd

meta = pd.read_csv(ROOT / "ocdd" / "metadata.csv")
print(meta.groupby(["split", "kind"]).size())
print(meta.groupby("level")[["regions", "coverage", "drift_ms_median", "drift_ms_p95"]].median())